## A UPSC Essay Grading Workflow (Parallel Workflow) 

In [21]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os 
from typing import TypedDict,Annotated
from pydantic import BaseModel, Field

In [2]:
load_dotenv()

True

In [31]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    timeout=30,
    max_retries=2,
)

In [32]:
class EvaluationSchema(BaseModel):
    feedback : str = Field(description="Detailed feedback for the essay")
    score : int = Field(description="Score out of 10", ge=0, le=10)
    

In [33]:
structured_model = llm.with_structured_output(EvaluationSchema)

In [34]:
essay = """**Role of India in AI**

India is emerging as a significant player in the global artificial intelligence landscape, driven by its vast talent pool, growing digital infrastructure, and supportive government policies. With one of the largest populations of STEM graduates and IT professionals in the world, India provides a strong human resource base for AI research, development, and deployment.

The Indian government has recognized AI's transformative potential and launched initiatives such as the National AI Strategy ("AI for All") and the establishment of Centres of Excellence in AI across sectors like healthcare, agriculture, and education. Programs like Digital India and the widespread adoption of Aadhaar and UPI have also created a massive digital data ecosystem, which is crucial for training AI models suited to local needs.

Indian startups and tech giants alike are actively contributing to AI innovation. Companies are developing AI solutions for regional languages, financial inclusion, precision farming, and affordable healthcare diagnostics—addressing challenges unique to a developing economy. Global technology firms have also set up major AI research labs in India, leveraging local talent for cutting-edge research.

Moreover, India plays an important role in the global AI workforce, with Indian engineers and researchers contributing significantly to AI development worldwide, both domestically and through diaspora communities in Silicon Valley and beyond.

However, challenges remain, including the need for greater investment in foundational research, data privacy frameworks, and equitable access to AI benefits across rural and urban divides.

In conclusion, India's role in AI is rapidly evolving from being a service provider to becoming an innovation hub, with the potential to shape inclusive and responsible AI development globally."""

In [35]:
prompt = f'Evaluate the language quality of the essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt)

EvaluationSchema(feedback='The essay effectively presents a coherent narrative about India\'s role in the AI landscape, with logical structure and clear argumentation. The introductory paragraph gives a solid overview, and each subsequent paragraph builds on this foundation. The use of specific examples, such as government initiatives and the capabilities of the Indian workforce, enriches the discussion and supports claims well. Additionally, the acknowledgment of challenges demonstrates a balanced perspective. However, some sentences could benefit from varied structure to enhance readability, and certain terms (e.g., "STEM graduates," "AI models") may confuse readers unfamiliar with the topic. Overall, the language is clear and precise, but some minor improvements in flow and complexity would elevate the quality further. ', score=8)

In [36]:
import operator

In [37]:
class UPSCState(TypedDict):
    essay :str
    language_feedback:str
    analysis_feedback :str
    clarity_feedback :str
    overall_feedback :str
    individual_scores : Annotated[list[int], operator.add]
    avg_score:float

In [38]:
def language(state: UPSCState) -> UPSCState:
    prompt = f'Evaluate the language quality of the essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'language_feedback' : output.feedback, "individual_scores" : [output.score]}

In [39]:
def analysis(state: UPSCState) -> UPSCState:
    prompt = f'Evaluate the depth of analysis of the essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback' : output.feedback, "individual_scores" : [output.score]}

In [40]:
def thought(state: UPSCState) -> UPSCState:
    prompt = f'Evaluate the clarity of thought of the essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback' : output.feedback, "individual_scores" : [output.score]}

In [44]:
def final(state: UPSCState) -> UPSCState:
    # summary feedack
    prompt = f'Based on the following feedbacks create a summarized feedback \n Language feedback - {state['language_feedback']} \n Depth of analysis feedback - {state['analysis_feedback']} \n Clarity of thought feedback {state['clarity_feedback']}'
    overall_feedback = llm.invoke(prompt).content

    # Avg Calculate
    avg_score = sum(state['individual_scores'])/ len(state['individual_scores'])

    return {'overall_feedback' : overall_feedback, 'avg_score' : avg_score}

In [45]:
graph = StateGraph(UPSCState)

graph.add_node('Evaluate_Language', language)
graph.add_node('Evaluate_analysis', analysis)
graph.add_node('Evaluate_Thought', thought)
graph.add_node('Final_Evaluation', final)

## Creating the Edges

graph.add_edge(START, 'Evaluate_Language')
graph.add_edge(START, 'Evaluate_analysis')
graph.add_edge(START, 'Evaluate_Thought')

graph.add_edge('Evaluate_Language', 'Final_Evaluation')
graph.add_edge('Evaluate_analysis', 'Final_Evaluation')
graph.add_edge('Evaluate_Thought', 'Final_Evaluation')

graph.add_edge('Final_Evaluation', END)

workflow = graph.compile()

In [46]:
initial_state = {
    'essay' : essay
}

workflow.invoke(initial_state)

{'essay': '**Role of India in AI**\n\nIndia is emerging as a significant player in the global artificial intelligence landscape, driven by its vast talent pool, growing digital infrastructure, and supportive government policies. With one of the largest populations of STEM graduates and IT professionals in the world, India provides a strong human resource base for AI research, development, and deployment.\n\nThe Indian government has recognized AI\'s transformative potential and launched initiatives such as the National AI Strategy ("AI for All") and the establishment of Centres of Excellence in AI across sectors like healthcare, agriculture, and education. Programs like Digital India and the widespread adoption of Aadhaar and UPI have also created a massive digital data ecosystem, which is crucial for training AI models suited to local needs.\n\nIndian startups and tech giants alike are actively contributing to AI innovation. Companies are developing AI solutions for regional languages